In [1]:
import torch
import shapelens as sl

## Simple self-attention (single head)
Every line is one step: `x` is projected to `q`, `k`, `v`; `q @ kᵀ` gives a `T × T` score for every pair of tokens; softmax turns scores into weights; the weights mix `v` back to `(B, T, D)`.

In [2]:
def attention(x, Wq, Wk, Wv, H):
    B, T, D = x.shape
    q = (x @ Wq).view(B, T, H, D // H).transpose(1, 2)
    k = (x @ Wk).view(B, T, H, D // H).transpose(1, 2)
    v = (x @ Wv).view(B, T, H, D // H).transpose(1, 2)
    scores = q @ k.transpose(-2, -1) / (D // H) ** 0.5
    w = scores.softmax(-1)
    return w @ v

In [3]:
B, T, D = 2, 5, 8                         # batch, tokens, embedding dim
x  = torch.randn(B, T, D, device="meta")  # meta: shapes only, no compute
Wq = torch.randn(D, D, device="meta")
Wk = torch.randn(D, D, device="meta")
Wv = torch.randn(D, D, device="meta")

with sl.trace() as t:
    q = x @ Wq
    k = x @ Wk
    v = x @ Wv
    scores = q @ k.transpose(-2, -1) / D ** 0.5   # every token against every token: (B, T, T)
    attn = scores.softmax(dim=-1)
    out = attn @ v                                # weighted sum of values: (B, T, D)
t

q = x @ Wq
    mm: x[B 2, T 5, D 8], Wq[D 8, D 8] -> q[B 2, T 5, D 8]
k = x @ Wk
    mm: x[B 2, T 5, D 8], Wk[D 8, D 8] -> k[B 2, T 5, D 8]
v = x @ Wv
    mm: x[B 2, T 5, D 8], Wv[D 8, D 8] -> v[B 2, T 5, D 8]
scores = q @ k.transpose(-2, -1) / D ** 0.5   # every token against every token: (B, T, T)
    transpose, bmm, div: k[B 2, T 5, D 8], q[B 2, T 5, D 8] -> scores[B 2, T 5, T 5]
attn = scores.softmax(dim=-1)
    _softmax: scores[B 2, T 5, T 5] -> attn[B 2, T 5, T 5]
out = attn @ v                                # weighted sum of values: (B, T, D)
    bmm: attn[B 2, T 5, T 5], v[B 2, T 5, D 8] -> out[B 2, T 5, D 8]

In [4]:
x = torch.randn(2, 5, 32, device="meta")  # meta: shapes only, no compute
W = [torch.randn(32, 32, device="meta") for _ in range(3)]

with sl.trace() as t:
    out = attention(x, *W, H=4)
t

q = (x @ Wq).view(B, T, H, D // H).transpose(1, 2)
    mm, transpose: x[B 2, T 5, D 32], Wq[D 32, D 32] -> q[B 2, H 4, T 5, D // H = 8]
k = (x @ Wk).view(B, T, H, D // H).transpose(1, 2)
    mm, transpose: x[B 2, T 5, D 32], Wk[D 32, D 32] -> k[B 2, H 4, T 5, D // H = 8]
v = (x @ Wv).view(B, T, H, D // H).transpose(1, 2)
    mm, transpose: x[B 2, T 5, D 32], Wv[D 32, D 32] -> v[B 2, H 4, T 5, D // H = 8]
scores = q @ k.transpose(-2, -1) / (D // H) ** 0.5
    transpose, bmm, div: k[B 2, H 4, T 5, D // H = 8], q[B 2, H 4, T 5, D // H = 8] -> scores[B 2, H 4, T 5, T 5]
w = scores.softmax(-1)
    _softmax: scores[B 2, H 4, T 5, T 5] -> w[B 2, H 4, T 5, T 5]
return w @ v
    bmm: w[B 2, H 4, T 5, T 5], v[B 2, H 4, T 5, D // H = 8] -> out[B 2, H 4, T 5, D // H = 8]

## Dimension labels
Sizes are named from your code automatically: UPPERCASE `int` variables (`B`, `T`, `D`, `H`), and integer arithmetic such as `D // H`, which shows as `D // H = 8` in the cell above.

To give an expression a short name, pass it with its exact code: `sl.trace(Dh="D // H")`. A size with no variable or expression can be named by value: `sl.trace(S=197)`.

In [ ]:
with sl.trace(Dh="D // H") as t:   # "Dh" names whatever `D // H` is in the code
    out = attention(x, *W, H=4)
t

q = (x @ Wq).view(B, T, H, D // H).transpose(1, 2)
    mm, transpose: x[B 2, T 5, D 32], Wq[D 32, D 32] -> q[B 2, H 4, T 5, Dh 8]
k = (x @ Wk).view(B, T, H, D // H).transpose(1, 2)
    mm, transpose: x[B 2, T 5, D 32], Wk[D 32, D 32] -> k[B 2, H 4, T 5, Dh 8]
v = (x @ Wv).view(B, T, H, D // H).transpose(1, 2)
    mm, transpose: x[B 2, T 5, D 32], Wv[D 32, D 32] -> v[B 2, H 4, T 5, Dh 8]
scores = q @ k.transpose(-2, -1) / (D // H) ** 0.5
    transpose, bmm, div: k[B 2, H 4, T 5, Dh 8], q[B 2, H 4, T 5, Dh 8] -> scores[B 2, H 4, T 5, T 5]
w = scores.softmax(-1)
    _softmax: scores[B 2, H 4, T 5, T 5] -> w[B 2, H 4, T 5, T 5]
return w @ v
    bmm: w[B 2, H 4, T 5, T 5], v[B 2, H 4, T 5, Dh 8] -> out[B 2, H 4, T 5, Dh 8]

In [6]:
# code defined in the cell itself must trace too
with sl.trace() as t2:
    a = torch.randn(3, 4, device="meta")
    b = torch.randn(4, 5, device="meta")
    c = a @ b
t2

a = torch.randn(3, 4, device="meta")
    randn:  -> a[3, 4]
b = torch.randn(4, 5, device="meta")
    randn:  -> b[4, T 5]
c = a @ b
    mm: a[3, 4], b[4, T 5] -> c[3, T 5]

## ViT patchify: image (B,C,H,W) -> tokens (B,N+1,D)

In [7]:
B, C, H, W, P, D = 2, 3, 224, 224, 16, 768   # batch, channels, height, width, patch size, embed dim
G = H // P                # patches per side
N = G * G                 # number of patches
K = P * P * C             # flattened patch length
img  = torch.randn(B, C, H, W, device="meta")
proj = torch.nn.Linear(K, D, device="meta")
cls  = torch.randn(1, 1, D, device="meta")
pos  = torch.randn(1, N + 1, D, device="meta")

with sl.trace() as vit:   # UPPERCASE ints above label the dims; extra: sl.trace(Z=123)
    grid    = img.reshape(B, C, G, P, G, P)          # cut H and W into (G, P) each
    perm    = grid.permute(0, 2, 4, 3, 5, 1)         # move the two patch-grid axes next to B
    patches = perm.reshape(B, N, K)                  # flatten each P x P x C patch -> N x K
    tokens  = proj(patches)                          # linear embed -> N x D
    seq     = torch.cat([cls.expand(B, -1, -1), tokens], dim=1)  # prepend [CLS] -> N+1
    z       = seq + pos                              # add position embedding
vit

grid    = img.reshape(B, C, G, P, G, P)          # cut H and W into (G, P) each
    view: img[B 2, C 3, W/H 224, W/H 224] -> grid[B 2, C 3, G 14, P 16, G 14, P 16]
perm    = grid.permute(0, 2, 4, 3, 5, 1)         # move the two patch-grid axes next to B
    permute: grid[B 2, C 3, G 14, P 16, G 14, P 16] -> perm[B 2, G 14, G 14, P 16, P 16, C 3]
patches = perm.reshape(B, N, K)                  # flatten each P x P x C patch -> N x K
    clone, _unsafe_view: perm[B 2, G 14, G 14, P 16, P 16, C 3] -> patches[B 2, N 196, D/K 768]
tokens  = proj(patches)                          # linear embed -> N x D
    t, addmm: patches[B 2, N 196, D/K 768] -> tokens[B 2, N 196, D/K 768]
seq     = torch.cat([cls.expand(B, -1, -1), tokens], dim=1)  # prepend [CLS] -> N+1
    cat: cls[1, 1, D/K 768], tokens[B 2, N 196, D/K 768] -> seq[B 2, 197, D/K 768]
z       = seq + pos                              # add position embedding
    add: seq[B 2, 197, D/K 768], pos[1, 197, D/K 768] -> z[B 2, 197, D/K 768]

## Shape bug: the failing op is highlighted
If an op fails inside the block, the trace is displayed right away (the cell's trailing `t` would never run), then the error still propagates.

In [8]:
B, T, D = 2, 5, 32
x = torch.randn(B, T, D, device="meta")
W_bad = torch.randn(64, 16, device="meta")  # should start with D = 32

try:
    with sl.trace():
        a = x + 1
        h = a @ W_bad  # 32 != 64
except RuntimeError as e:
    print("error still propagates:", str(e).splitlines()[0])

a = x + 1
    add: x[B 2, T 5, D 32] -> a[B 2, T 5, D 32]
✗ h = a @ W_bad  # 32 != 64
    mm: a[B 2, T 5, D 32], W_bad[64, P 16] -> ERROR: a and b must have same reduction dim, but got [10, 32] X [64, 16].

error still propagates: a and b must have same reduction dim, but got [10, 32] X [64, 16].
